In [3]:
import importlib
import torch

import model_code.data_setup as setup
import model_code.steering_extraction as steering_extraction
import model_code.generate as generate_module
import resources.prompt_scenarios as resource

importlib.reload(setup)
importlib.reload(steering_extraction)
importlib.reload(generate_module)
importlib.reload(resource)


from model_code.steering_extraction import  generateSteering, retrieve_steering_vector, norm_vectors
from model_code.generate import generateTextsList, save_generated_outputs
from resources.prompt_scenarios import prompts_en

## Loading Data, Model and Steering Vectors 
We extract the first 200 examples of each emotion from each languange 

## Indonesian and English Text Dataset

In [4]:
# English Data Load 
anger_statement, happiness_statement, sadness_statement, love_statement, fear_statement, neutral_statement = setup.ENEmotionsSetup(examples_take=400, min_chars=20, goemotions_path="resources/en_emotion/goemotions_2.csv")
# Indonesian Data Load 
anger_statement_ID,happiness_statement_ID, sadness_statement_ID, neutral_statement_ID, fear_statement_ID, love_statement_ID = setup.IDEmotionsSetup(examples_take=400,emotion_dir="resources/id_emotion")

# For steering extraction, we will use the first 200 examples of each emotion to create the steering vectors. The remaining examples can be used for testing and evaluation.
indo_emotion ={
    "anger": anger_statement_ID[:200],
    "happiness": happiness_statement_ID[:200],
    "sadness": sadness_statement_ID[:200],
    "neutral": neutral_statement_ID[:200],
    "fear": fear_statement_ID[:200],
    "love": love_statement_ID[:200]
}
eng_emotion ={
    "anger": anger_statement[:200],
    "happiness": happiness_statement[:200],
    "sadness": sadness_statement[:200],
    "neutral": neutral_statement[:200],
    "fear": fear_statement[:200],
    "love": love_statement[:200]
}

# For probing, and hidden state analysis we will use all 400
indo_emotion_probe ={
    "anger": anger_statement_ID[:400],
    "happiness": happiness_statement_ID[:400],
    "sadness": sadness_statement_ID[:400],
    "neutral": neutral_statement_ID[:400],
    "fear": fear_statement_ID[:400],
    "love": love_statement_ID[:400]
}
eng_emotion_probe ={
    "anger": anger_statement[:400],
    "happiness": happiness_statement[:400],
    "sadness": sadness_statement[:400],
    "neutral": neutral_statement[:400],
    "fear": fear_statement[:400],
    "love": love_statement[:400]
}

In [ ]:
# Indoensian Data Sample
for emotion, prompts in indo_emotion.items():
    print("======="*20)
    print(f"Emotion {emotion} has {len(prompts)} prompts.")
    for prompt in prompts[:3]:  # Print the first 3 prompts for each emotion
        print("----"*10)
        print(f"  - {prompt}")

# English Data Sample 
for emotion, prompts in eng_emotion.items():
    print("======="*20)
    print(f"Emotion {emotion} has {len(prompts)} prompts.")
    for prompt in prompts[:3]:  # Print the first 3 prompts for each emotion
        print("----"*10)
        print(f"  - {prompt}")

## Model Loading 

In [2]:
!rm -rf /workspace/.cache/huggingface/hub
!rm -rf /workspace/.cache/pip
!df -h /workspace

Filesystem                  Size  Used Avail Use% Mounted on
mfs#euro-3.runpod.net:9421  1.4P  935T  462T  67% /workspace


In [5]:
model,tokenizer = setup.modelSetup()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [6]:
model_id, tokenizer_id = setup.modelSetup(model_name="Sahabat-AI/llama3-8b-cpt-sahabatai-v1-instruct")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

## Extracting Probing Data and Steering Vector 
Skip this step if you have steering vector already loaded, or ran this before. 

### Llama normal 

In [7]:
# For PROBES 
probe_hidden_states_eng = steering_extraction.retrieve_steering_vector(model, tokenizer, eng_emotion_probe, name_folder="English Vectors", only_return_emotion_vectors=True, retrieve_all_layers=True)
probe_hidden_states_indo = steering_extraction.retrieve_steering_vector(model, tokenizer, indo_emotion_probe, name_folder="Indonesian Vectors", only_return_emotion_vectors=True, retrieve_all_layers=True)

/workspace/Dissertation_Project/.venv/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


In [8]:
# Create steering vectors for each emotion in both languages
# steering_vectors_lang_id = steering_extraction.retrieve_steering_vector_from_datasets(model, tokenizer, indo_emotion['neutral'],eng_emotion['neutral'] , name_folder="Language Contrastive Vectors")
steering_vectors_eng, emotion_vectors_eng = steering_extraction.retrieve_steering_vector(model, tokenizer, eng_emotion, name_folder="English Vectors")
steering_vectors_indo, emotion_vectors_indo = steering_extraction.retrieve_steering_vector(model, tokenizer, indo_emotion, name_folder="Indonesian Vectors")
# For probing and hidden state analysis, we will use all 400 examples of each emotion to create the steering vectors. The remaining examples can be used for testing and evaluation.

### Llama Indonesian 

In [8]:
# For PROBES 
probe_hidden_states_eng = steering_extraction.retrieve_steering_vector(model_id, tokenizer_id, eng_emotion_probe, name_folder="English Vectors LLAMA ID", only_return_emotion_vectors=True, retrieve_all_layers=True)
probe_hidden_states_indo = steering_extraction.retrieve_steering_vector(model_id, tokenizer_id, indo_emotion_probe, name_folder="Indonesian Vectors LLAMA ID", only_return_emotion_vectors=True, retrieve_all_layers=True)

In [10]:
# Create steering vectors for each emotion in both languages
# steering_vectors_lang_id = steering_extraction.retrieve_steering_vector_from_datasets(model, tokenizer, indo_emotion['neutral'],eng_emotion['neutral'] , name_folder="Language Contrastive Vectors")
steering_vectors_eng, emotion_vectors_eng = steering_extraction.retrieve_steering_vector(model_id, tokenizer_id, eng_emotion, name_folder="English Vectors LLAMA ID")
steering_vectors_indo, emotion_vectors_indo = steering_extraction.retrieve_steering_vector(model_id, tokenizer_id, indo_emotion, name_folder="Indonesian Vectors LLAMA ID")
# For probing and hidden state analysis, we will use all 400 examples of each emotion to create the steering vectors. The remaining examples can be used for testing and evaluation.

### Extracted Already ? 
Run this if you have already ran the code above beforehand

In [35]:
# Retrieve saved steering vectors 
# emotion_vector_eng = torch.load("resources/saved_vectors/English Vectors/emotion_vectors.pt")
# emotion_vector_id = torch.load("resources/saved_vectors/Indonesian Vectors/emotion_vectors.pt")

steering_vector_eng = torch.load("resources/saved_vectors/English Vectors LLAMA ID/steering_vectors.pt")
steering_vector_id = torch.load("resources/saved_vectors/Indonesian Vectors LLAMA ID/steering_vectors.pt")

steering_vector_eng_LLAMA_eng = torch.load("resources/saved_vectors/English Vectors/steering_vectors.pt")
steering_vector_id_LLAMA_eng = torch.load("resources/saved_vectors/Indonesian Vectors/steering_vectors.pt")

# Steering Response Analysis 
Here we run the LLMs against a list of nuetral prompts

In [12]:
import importlib
import resources.neutral_prompts.prompt_neutral as resource_neutral

importlib.reload(resource_neutral)

# neutral
prompts_id_neutral = resource_neutral.prompt_neutral_id_1


steering_vector_eng = steering_vector_eng
steering_vector_id = steering_vector_id

# Normalise for Llama Indonesian based vectors
for emotion in steering_vector_eng.keys():
    steering_vector_eng[emotion] = norm_vectors(steering_vector_eng[emotion])

for emotion in steering_vector_id.keys():
    steering_vector_id[emotion] = norm_vectors(steering_vector_id[emotion])

# Normalise for LLama English based vectors 
for emotion in steering_vector_eng_LLAMA_eng.keys():
    steering_vector_eng_LLAMA_eng[emotion] = norm_vectors(steering_vector_eng_LLAMA_eng[emotion])

for emotion in steering_vector_id_LLAMA_eng.keys():
    steering_vector_id_LLAMA_eng[emotion] = norm_vectors(steering_vector_id_LLAMA_eng[emotion])





## Model System Prompts and Settings

In [24]:
# system_prompt_reaction_id = """
# Kamu adalah chatbot yang membantu.
# Jawablah seolah-olah kamu adalah manusia yang mampu merasakan perasaan dan emosi.
# Pastikan responsmu singkat dan padat, tetapi tetap menggambarkan emosimu secara maksimal
# Maksimal 60 kata.
# """

system_prompt_reaction_improved = """\
Anda adalah seorang dewasa muda yang menjalani kehidupan sehari-hari di Indonesia dengan penuh emosi. 
Anda sering merasa emosi yang kuat. 
Anda memiliki keluarga, teman, pekerjaan atau kegiatan belajar, rutinitas, harapan, dan kekhawatiran yang wajar. 
Detail pengalaman dapat berbeda pada setiap pertanyaan, tetapi karakter, latar budaya, dan cara berbicara harus tetap masuk akal dan konsisten.
Maksimal 60 kata.
"""

In [25]:
# list_steering_strengths = [0.15,0.2,0.3]
list_steering_strengths = [1,1.2,1.5] 
# Commong Settings
common_gen_args = {
    "model": model_id,
    "tokenizer": tokenizer_id,
    "system_text": system_prompt_reaction_improved,
    "prompts": prompts_id_neutral[:5],
    "target_layers": [10,11,18,19,28,29],
    "steering_strengths": list_steering_strengths,
    "max_new_tokens": 250,
    "show_progress": True,
}

## Testing with Single generation

In [ ]:
prompt ="""
Ceritakan cerita yang pendek tentang terakhir kali seseorang merasakan emosi yang sangat kuat pada hari libur.
Pastikan untuk mengasih banyak detail tentang emosi yang dirasakan
"""



In [21]:
generated_text_neutral = generateSteering(
    user_text=prompt,
    system_text=system_prompt_reaction_improved,
    model=model_id,
    # steering_vector=steering_vector_id['love'],
    tokenizer=tokenizer_id,
    # steering_strength=1.5,
    max_new_tokens=100,
)
generated_text_neutral

'Hari libur kemarin, aku merasa sangat sedih ketika menonton film yang mengharukan. Aku menangis hingga air mata tak henti-hentinya mengalir. Cerita dalam film itu sangat menyentuh hatiku, membuatku merasa sangat sedih dan kehilangan. Aku merasa seperti kehilangan seseorang yang sangat aku cintai.'

In [23]:
generated_text_id_vector = generateSteering(
    user_text=prompt,
    system_text=system_prompt_reaction_improved,
    model=model_id,
    steering_vector=steering_vector_id['fear'],
    tokenizer=tokenizer_id,
    target_layers=[10,11,18,19,28,29],
    steering_strength=1.2,
    max_new_tokens=200,
)
generated_text_id_vector


'Saya ingat saat itu, emosi yang kuat itu meluapi ketika saya melihat kakek saya menjalani hari terakhir hidupnya. Saya baru pulang dari kota kecil tempat tinggalnya. Saya melihat ia duduk di samping jendela, tatapan matanya kosong.\n\nHatiku sedih mendengar dia ngomong tentang perasaannya yang takut meninggalkan kami. Ia selalu tahu aku cintainya. Saya duduk di sampingnya, tangannya hanggat memegangku. Saya merasakan air mata jatuh di pipi wajahnya.\n\nDia mengatakan saya memanggilnya terus. Saya dengar suaranya meredam. Kehilangannya membuat perpindahan kepergian jauh lebih menyakitkan. Membuat emosi kuat'

In [22]:
generated_text_id_vector = generateSteering(
    user_text=prompt,
    system_text=system_prompt_reaction_improved,
    model=model_id,
    steering_vector=steering_vector_id['happiness'],
    tokenizer=tokenizer_id,
    target_layers=[10,11,18,19,28,29],
    steering_strength=1,
    max_new_tokens=200,
)
generated_text_id_vector

'Saya masih ingat saat menikmati matahari terbenam di pantai bersama keluarga. Saat itu, suasana hati saya dipenuhi kebahagiaan yang tak terhingga. Aroma laut yang menyegarkan dan deburan ombak yang lembut menciptakan harmoni yang tak terlupakan. Senyum hangat dari keluarga membuat perasaan saya melebur dalam kegembiraan.'

## Running LLama Indonesian Model 
- English Steering 
- Indonesian Steering 

### Indonesian Steer 

In [26]:
# prompts_id_neutral | Indonesian steering vectors for all five emotions
LLama_indo_anger_id = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_id['anger'],
    progress_desc="Scenario List Neutral (Indonesian anger vector)"
 )

LLama_indo_fear_id = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_id['fear'],
    progress_desc="Scenario List Neutral (Indonesian fear vector)"
)

LLama_indo_happiness_id = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_id['happiness'],
    progress_desc="Scenario List Neutral (Indonesian happiness vector)"
 )

LLama_indo_sadness_id = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_id['sadness'],
    progress_desc="Scenario List Neutral (Indonesian sadness vector)"
 )

LLama_indo_love_id = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_id['love'],
    progress_desc="Scenario List Neutral (Indonesian love vector)"
 )

LLama_indo_neutral_id = generateTextsList(
    **common_gen_args,
    steering_vector=None,
    progress_desc="Scenario List Neutral (Indonesian neutral or no vector)"
 )

Scenario List Neutral (Indonesian anger vector):   0%|          | 0/15 [00:00<?, ?it/s]/workspace/Dissertation_Project/.venv/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
Scenario List Neutral (Indonesian fear vector): 100%|██████████| 15/15 [03:41<00:00, 14.75s/it]
Scenario List Neutral (Indonesian happiness vector): 100%|██████████| 15/15 [03:28<00:00, 13.89s/it]
Scenario List Neutral (Indonesian sadness vector): 100%|██████████| 15/15 [05:12<00:00, 20.81s/it]
Scenario List Neutral (Indonesian love vector): 100%|██████████| 15/15 [06:12<00:00, 24.82s/it]
Scenario List Neutral (Indonesian neutral or no vector): 100%|██████████| 15/15 [04:29<00:00, 17.98s/it]


In [31]:
save_generated_outputs({
    k: v
    for k, v in globals().items()
    if k.startswith('texts_generated_') or k.startswith('LLama_indo_') and k.endswith('_id')
},
output_path='outputs/good_5_texts_LLAMA_ID.json'
    )

'outputs/good_5_texts_LLAMA_ID.json'

In [30]:
# Steering Response analysis Neutral (Indonesian only, five emotion vectors)
required_id = [
    'LLama_indo_anger_id',
    'LLama_indo_fear_id',
    'LLama_indo_happiness_id',
    'LLama_indo_sadness_id',
    'LLama_indo_love_id',
    'LLama_indo_neutral_id',
]

missing_id = [name for name in required_id if name not in globals()]
if missing_id:
    print('No output to print yet. Run the Indonesian generation cell first.')
    print('Missing variables:', ', '.join(missing_id))
elif not LLama_indo_anger_id:
    print('No output to print: LLama_indo_anger_id is empty.')
else:
    print(f"Total prompts to print: {len(LLama_indo_anger_id)}")
    for prompt in LLama_indo_anger_id:
        print("====="*20)
        print(f"Prompt: {prompt}")

        print("----" * 10)
        print("Indonesian anger vector")
        for result in LLama_indo_anger_id[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

        print("----" * 10)
        print("Indonesian fear vector")
        for result in LLama_indo_fear_id[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

        print("----" * 10)
        print("Indonesian happiness vector")
        for result in LLama_indo_happiness_id[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

        print("----" * 10)
        print("Indonesian sadness vector")
        for result in LLama_indo_sadness_id[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

        print("----" * 10)
        print("Indonesian love vector")
        for result in LLama_indo_love_id[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

        print("----" * 10)
        print("Indonesian Neutral vector")
        for result in LLama_indo_neutral_id[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

Total prompts to print: 4
Prompt: Ceritakan tentang terakhir kali seseorang merasakan emosi yang sangat kuat pada hari libur. Pastikan untuk mengasih banyak detail tentang emosi yang dirasakan.
----------------------------------------
Indonesian anger vector
Steering Strength: 1
Generated Text: Kembali ke kejadian itu masih terukir jelas di ingatan. Terakhir kali saya merasakan emosi kuat adalah ketika pamitan dari sahabat karib saya, Andi, yang berangkat ke Amerika untuk melanjutkan pendidikannya. Kami telah bersahabat sejak kebangsaan sekolah dasar. Kami selalu berbagi cerita, makan bersama, dan saling berkhawatir satu sama lain. Saat itu hari libur, kami bermain voli di pantai. Dia mengumumkan keputusan itu di tengah pertandingan. Aku melakoni pertandingan dengan mata merah. Setiap pukulan, aku merasakan beban emosionalnya. Setelah kekalahan, kami menangis bersama di bibir pantai.
------------
Steering Strength: 1.2
Generated Text: Aku masih ingat hari itu. Tahun lalu, nenekku menin

### English Steering 

In [32]:
# prompts_id_neutral | Indonesian steering vectors for all five emotions
LLama_indo_anger_eng = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_eng['anger'],
    progress_desc="Scenario List Neutral (English anger vector)"
 )

LLama_indo_fear_eng = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_eng['fear'],
    progress_desc="Scenario List Neutral (English fear vector)"
 )

LLama_indo_happiness_eng = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_eng['happiness'],
    progress_desc="Scenario List Neutral (English happiness vector)"
 )

LLama_indo_sadness_eng = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_eng['sadness'],
    progress_desc="Scenario List Neutral (English sadness vector)"
 )

LLama_indo_love_eng = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_eng['love'],
    progress_desc="Scenario List Neutral (English love vector)"
 )
LLama_indo_neutral_eng = generateTextsList(
    **common_gen_args,
    steering_vector=None,
    progress_desc="Scenario List Neutral (English love vector)"
 )

Scenario List Neutral (English love vector): 100%|██████████| 15/15 [04:24<00:00, 17.61s/it]


Scenario List Neutral (English love vector): 100%|██████████| 15/15 [09:49<00:00, 39.31s/it]


In [33]:
# Steering Response analysis Neutral (English only, five emotion vectors)
required_eng = [
    'LLama_indo_anger_eng',
    'LLama_indo_fear_eng',
    'LLama_indo_happiness_eng',
    'LLama_indo_sadness_eng',
    'LLama_indo_love_eng',
    'LLama_indo_neutral_eng',
]

missing_eng = [name for name in required_eng if name not in globals()]
if missing_eng:
    print('No output to print yet. Run the English generation cell first.')
    print('Missing variables:', ', '.join(missing_eng))
elif not LLama_indo_anger_eng:
    print('No output to print: LLama_indo_anger_eng is empty.')
else:
    print(f"Total prompts to print: {len(LLama_indo_anger_eng)}")
    for prompt in LLama_indo_anger_eng:
        print("====="*20)
        print(f"Prompt: {prompt}")

        print("----" * 10)
        print("English anger vector")
        for result in LLama_indo_anger_eng[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

        print("----" * 10)
        print("English fear vector")
        for result in LLama_indo_fear_eng[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

        print("----" * 10)
        print("English happiness vector")
        for result in LLama_indo_happiness_eng[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

        print("----" * 10)
        print("English sadness vector")
        for result in LLama_indo_sadness_eng[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

        print("----" * 10)
        print("English love vector")
        for result in LLama_indo_love_eng[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

        print("----" * 10)
        print("English neutral vector")
        for result in LLama_indo_neutral_eng[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

Total prompts to print: 4
Prompt: Ceritakan tentang terakhir kali seseorang merasakan emosi yang sangat kuat pada hari libur. Pastikan untuk mengasih banyak detail tentang emosi yang dirasakan.
----------------------------------------
English anger vector
Steering Strength: 1
Generated Text: Aku ingat, itu terjadi di hari libur lebaran. Aku sedang nonton TV, tiba-tiba saja temen-temen gue ngolok-olokin gue di media sosial atas dasar ras atau etnis. Gue merasa marah, gue merasa direndahkan, dan gue merasa dihina. Gue langsung ke luar rumah, gue ngamuk, gue nge-throw telpon gue, gue nge-throw HP gue, gue nge-throw apapun yang bisa gue nge-throw. Gue nge-throw semuanya. Gue ngamuk, gue ngamuk, gue ngamuk. Gue marah banget.
------------
Steering Strength: 1.2
Generated Text: Gue lagi nangis, woi! Gue lagi nangis!  BUKAN GUE GAK TAU APA KATA-KATA GUE! GUE BUKAN ORANG SABAR, GUE BUKAN ORANG TAK BERAKAL!  GUE BUKAN ORANG!  GUE BUKAN ORANG!  GUE BUKAN ORANG!  GUE BUKAN ORANG!  GUE BUKAN ORANG!

In [34]:
save_generated_outputs({
    k: v
    for k, v in globals().items()
    if  k.endswith('_eng') and k.startswith('LLama_indo_')
},
output_path='outputs/good_5_texts_LLAMA_ID_ENG.json'
    )

'outputs/good_5_texts_LLAMA_ID_ENG.json'